# Step 21: Advanced: Provider Strategy and Production Hardening

        _Instructor Solution Notebook_  
        Level: **Advanced**  
        Estimated time: **110 minutes**

        ## Learning Objectives
        - [ ] Reason about provider selection as a product and operations decision.
- [ ] Design fallback and degradation behavior explicitly.
- [ ] Track latency, rough cost, and failure categories together.
- [ ] Create a hardening checklist for production-ready agent applications.

        ## Prerequisites
        - Core Step 15 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

This notebook turns production readiness into concrete decision-making. Instead of only measuring one call, you will reason about provider strategy, failure handling, and graceful degradation.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Model provider policies


In [ ]:
provider_policy = {
    "primary": "openrouter",
    "secondary": "azure_openai",
    "max_latency_seconds": 8.0,
    "max_estimated_cost_usd": 0.05,
    "fallback_on_errors": {"timeout", "rate_limit", "temporary_unavailable"},
}

print_json(provider_policy, title="Provider policy")


## Part 2: Core Implementation

### Capture an execution record shape


In [ ]:
def execution_record(provider: str, latency: float, estimated_cost: float, status: str) -> dict:
    return {
        "provider": provider,
        "latency_seconds": latency,
        "estimated_cost_usd": estimated_cost,
        "status": status,
    }

records = [
    execution_record("openrouter", 2.1, 0.004, "ok"),
    execution_record("azure_openai", 6.8, 0.011, "ok"),
    execution_record("openrouter", 9.4, 0.004, "timeout"),
]
print_json(records, title="Execution records")


## Part 2: Core Implementation

### Design fallback logic explicitly


In [ ]:
def should_fallback(record: dict, policy: dict) -> bool:
    if record["status"] in policy["fallback_on_errors"]:
        return True
    if record["latency_seconds"] > policy["max_latency_seconds"]:
        return True
    return False

fallback_decisions = [should_fallback(record, provider_policy) for record in records]
print_json(fallback_decisions, title="Fallback decisions")


## Part 2: Core Implementation

### Build a production hardening checklist


In [ ]:
hardening_checklist = [
    "Define provider failure categories and fallback rules.",
    "Log latency, estimated token usage, and status per request.",
    "Add health checks for provider credentials and endpoint reachability.",
    "Document degraded-mode behavior for operators and users.",
    "Review prompt, tool, and workflow changes with evaluation traces.",
]
print_json(hardening_checklist, title="Hardening checklist")


## Part 3: Hands-On Exercises

### Exercise 1
Add a degraded-mode response strategy for when both primary and secondary providers fail.

### Exercise 2
Write a helper that classifies each execution record as healthy, degraded, or failed.

This solution notebook includes completed code for both guided exercises.


In [ ]:
def degraded_mode_response(user_request: str) -> str:
    return (
        "The live providers are temporarily unavailable. "
        "Return a cached explanation, queue the request for retry, or ask the user to try again later."
    )

print_json(
    {"degraded_mode": degraded_mode_response("Explain tool orchestration tradeoffs.")},
    title="Exercise 1 solution",
)


In [ ]:
def classify_runtime_state(record: dict, policy: dict) -> str:
    if record["status"] != "ok":
        return "failed"
    if record["latency_seconds"] > policy["max_latency_seconds"] or record["estimated_cost_usd"] > policy["max_estimated_cost_usd"]:
        return "degraded"
    return "healthy"

print_json(
    [classify_runtime_state(record, provider_policy) for record in records],
    title="Exercise 2 solution",
)


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
def degraded_mode_response(user_request: str) -> str:
    return (
        "The live providers are temporarily unavailable. "
        "Return a cached explanation, queue the request for retry, or ask the user to try again later."
    )

print_json(
    {"degraded_mode": degraded_mode_response("Explain tool orchestration tradeoffs.")},
    title="Exercise 1 solution",
)


In [ ]:
def classify_runtime_state(record: dict, policy: dict) -> str:
    if record["status"] != "ok":
        return "failed"
    if record["latency_seconds"] > policy["max_latency_seconds"] or record["estimated_cost_usd"] > policy["max_estimated_cost_usd"]:
        return "degraded"
    return "healthy"

print_json(
    [classify_runtime_state(record, provider_policy) for record in records],
    title="Exercise 2 solution",
)


## Part 5: Best Practices & Tips

        - Treat provider choice as an operating policy, not only a config flag.
- Define degraded behavior before failures happen in production.
- Track cost, latency, and status together so tradeoffs are visible.
- Revisit hardening rules whenever prompts, tools, or workflows change materially.


## Reflection & Next Experiments

- Which part of `Advanced: Provider Strategy and Production Hardening` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You extended the production story from basic metrics to explicit provider strategy, degradation handling, and operational hardening.


## Additional Resources

        - `core notebook: 15_production_features.ipynb`
- `docs/llm_providers.md`
- `docs/depth_expansion_plan.md`
